# Exploratory Data Analysis
Reproducible exploration of the synthetic users, events, and transactions. The data is synthetic by design; this notebook checks scale, distributions, time coverage, and acquisition mix rather than presenting it as real company evidence.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
users = pd.read_csv(ROOT/'data/raw/users.csv', parse_dates=['signup_date'])
events = pd.read_csv(ROOT/'data/raw/events.csv', parse_dates=['event_time'])
orders = pd.read_csv(ROOT/'data/raw/transactions.csv', parse_dates=['order_time'])
orders['net_revenue'] = orders['amount'] - orders['refund_amount']
{'users': len(users), 'events': len(events), 'orders': len(orders), 'net_revenue': orders.net_revenue.sum()}

## Data quality and coverage
Check key uniqueness, missingness, date ranges, and the basic revenue distribution before interpreting downstream metrics.

In [ ]:
quality = pd.DataFrame({
    'rows': [len(users), len(events), len(orders)],
    'duplicate_keys': [users.user_id.duplicated().sum(), events.event_id.duplicated().sum(), orders.order_id.duplicated().sum()],
    'missing_cells': [users.isna().sum().sum(), events.isna().sum().sum(), orders.isna().sum().sum()]
}, index=['users','events','orders'])
display(quality)
display(orders.net_revenue.describe(percentiles=[.5,.75,.9,.95,.99]).to_frame('net_revenue'))

## Acquisition and device mix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
users.channel.value_counts().plot.bar(ax=axes[0], title='Users by acquisition channel')
users.device.value_counts().plot.bar(ax=axes[1], title='Users by device')
for ax in axes: ax.set_ylabel('Users'); ax.grid(axis='y', alpha=.2)
plt.tight_layout()

## Monthly activity and revenue
This view makes gaps, seasonality, and abrupt generator artifacts visible before KPI modeling.

In [ ]:
monthly_events = events.set_index('event_time').resample('MS').size().rename('events')
monthly_revenue = orders.set_index('order_time').resample('MS').net_revenue.sum().rename('net_revenue')
trend = pd.concat([monthly_events, monthly_revenue], axis=1)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
trend.events.plot(ax=axes[0], marker='o', title='Monthly event volume')
trend.net_revenue.plot(ax=axes[1], marker='o', color='#2a9d8f', title='Monthly net revenue')
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()

## Event funnel proxy
Counts are descriptive interaction volumes, not a causal conversion funnel because users may repeat events and enter at different stages.

In [ ]:
event_counts = events.event_type.value_counts().reindex(['browse','search','add_to_cart','purchase','login']).dropna()
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(event_counts.index.astype(str), event_counts.values, color='#457b9d')
ax.set(title='Event volume by type', ylabel='Events'); ax.grid(axis='y', alpha=.2)
plt.tight_layout()